In [41]:
# ============================================================
# NOTEBOOK 03.2 — SYNTHETIC INVESTOR POPULATION
# ============================================================

import os
import numpy as np
import pandas as pd

from pathlib import Path

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\NudgeIQ")

DATA_DIR = PROJECT_ROOT / "data" / "processed"

N_INVESTORS = 11162

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("=" * 70)
print("NudgeIQ — NOTEBOOK 03.2")
print("SYNTHETIC INVESTOR POPULATION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nData directory:")
print(DATA_DIR)

print("\nSynthetic population size:")
print(N_INVESTORS)

print("\nRandom seed:")
print(RANDOM_SEED)

NudgeIQ — NOTEBOOK 03.2
SYNTHETIC INVESTOR POPULATION

Project root:
D:\NudgeIQ

Data directory:
D:\NudgeIQ\data\processed

Synthetic population size:
11162

Random seed:
42


In [42]:
# ============================================================
# CELL 2 — EMPIRICAL JOINT DISTRIBUTION
# ============================================================

joint_distribution = pd.DataFrame({

    "Financial_Level": [
        "Low","Low","Low",
        "Low","Low","Low",
        "Low","Low","Low",

        "Medium","Medium","Medium",
        "Medium","Medium","Medium",
        "Medium","Medium","Medium",

        "High","High","High",
        "High","High","High",
        "High","High","High"
    ],

    "Engagement_Level": [
        "Low","Low","Low",
        "Medium","Medium","Medium",
        "High","High","High",

        "Low","Low","Low",
        "Medium","Medium","Medium",
        "High","High","High",

        "Low","Low","Low",
        "Medium","Medium","Medium",
        "High","High","High"
    ],

    "Profile_Level": [
        "Low","Medium","High",
        "Low","Medium","High",
        "Low","Medium","High",

        "Low","Medium","High",
        "Low","Medium","High",
        "Low","Medium","High",

        "Low","Medium","High",
        "Low","Medium","High",
        "Low","Medium","High"
    ],

    "Customers": [
        # LOW financial
        634, 350, 66,
        405, 573, 218,
        104, 344, 386,

        # MEDIUM financial
        1168, 948, 322,
        693, 1180, 697,
        147, 461, 724,

        # HIGH financial
        263, 262, 143,
        149, 269, 283,
        26, 124, 223
    ]
})

print("=" * 70)
print("EMPIRICAL JOINT DISTRIBUTION")
print("=" * 70)

print("\nNumber of cells:",
      len(joint_distribution))

print("\nTotal:",
      joint_distribution["Customers"].sum())

display(joint_distribution)

EMPIRICAL JOINT DISTRIBUTION

Number of cells: 27

Total: 11162


,Financial_Level,Engagement_Level,Profile_Level,Customers
0,Low,Low,Low,634
1,Low,Low,Medium,350
2,Low,Low,High,66
3,Low,Medium,Low,405
4,Low,Medium,Medium,573
5,Low,Medium,High,218
6,Low,High,Low,104
7,Low,High,Medium,344
8,Low,High,High,386
9,Medium,Low,Low,1168


In [43]:
# ============================================================
# CELL 3 — DISTRIBUTION VALIDATION
# ============================================================

joint_distribution["Probability"] = (
    joint_distribution["Customers"]
    / joint_distribution["Customers"].sum()
)

print("=" * 70)
print("EMPIRICAL DISTRIBUTION VALIDATION")
print("=" * 70)

print(
    "\nTotal customers:",
    joint_distribution["Customers"].sum()
)

print(
    "Probability total:",
    joint_distribution["Probability"].sum()
)

print(
    "\nMinimum probability:",
    joint_distribution["Probability"].min()
)

print(
    "Maximum probability:",
    joint_distribution["Probability"].max()
)

assert len(joint_distribution) == 27
assert joint_distribution["Customers"].sum() == N_INVESTORS
assert np.isclose(
    joint_distribution["Probability"].sum(),
    1.0
)

print("\nVALIDATION PASSED")

EMPIRICAL DISTRIBUTION VALIDATION

Total customers: 11162
Probability total: 1.0000000000000002

Minimum probability: 0.0023293316609926534
Maximum probability: 0.1057158215373589

VALIDATION PASSED


In [44]:
# ============================================================
# CELL 4 — GENERATE FICTIONAL INVESTORS
# ============================================================

synthetic_cells = joint_distribution[
    [
        "Financial_Level",
        "Engagement_Level",
        "Profile_Level"
    ]
].copy()

probabilities = (
    joint_distribution["Probability"]
    .values
)

sampled_indices = np.random.choice(
    len(synthetic_cells),
    size=N_INVESTORS,
    replace=True,
    p=probabilities
)

synthetic_investors = (
    synthetic_cells
    .iloc[sampled_indices]
    .reset_index(drop=True)
)

print("=" * 70)
print("SYNTHETIC INVESTOR POPULATION CREATED")
print("=" * 70)

print("\nRows:",
      len(synthetic_investors))

print("\nColumns:")
print(list(synthetic_investors.columns))

display(
    synthetic_investors.head(10)
)

SYNTHETIC INVESTOR POPULATION CREATED

Rows: 11162

Columns:
['Financial_Level', 'Engagement_Level', 'Profile_Level']


,Financial_Level,Engagement_Level,Profile_Level
0,Medium,Low,Low
1,High,Medium,High
2,Medium,High,Low
3,Medium,Medium,Medium
4,Low,Medium,Medium
5,Low,Medium,Medium
6,Low,Low,Medium
7,High,Low,Low
8,Medium,Medium,Medium
9,Medium,Medium,High


In [45]:
# ============================================================
# CELL 5 — SYNTHETIC CUSTOMER IDs
# ============================================================

synthetic_investors.insert(
    0,
    "Synthetic_Customer_ID",
    [
        f"SYN{i:05d}"
        for i in range(
            1,
            len(synthetic_investors) + 1
        )
    ]
)

print("=" * 70)
print("SYNTHETIC CUSTOMER IDs CREATED")
print("=" * 70)

print(
    "\nUnique IDs:",
    synthetic_investors[
        "Synthetic_Customer_ID"
    ].nunique()
)

display(
    synthetic_investors.head(10)
)

SYNTHETIC CUSTOMER IDs CREATED

Unique IDs: 11162


,Synthetic_Customer_ID,Financial_Level,Engagement_Level,Profile_Level
0,SYN00001,Medium,Low,Low
1,SYN00002,High,Medium,High
2,SYN00003,Medium,High,Low
3,SYN00004,Medium,Medium,Medium
4,SYN00005,Low,Medium,Medium
5,SYN00006,Low,Medium,Medium
6,SYN00007,Low,Low,Medium
7,SYN00008,High,Low,Low
8,SYN00009,Medium,Medium,Medium
9,SYN00010,Medium,Medium,High


In [46]:
# ============================================================
# CELL 6 — CONFIG-DRIVEN PERSONA ASSIGNMENT
# ============================================================

persona_rules_path = (
    DATA_DIR / "persona_rules.csv"
)

print("=" * 70)
print("LOADING PERSONA RULE CATALOGUE")
print("=" * 70)

print("\nPath:")
print(persona_rules_path)

if not persona_rules_path.exists():
    raise FileNotFoundError(
        f"Persona rules not found: {persona_rules_path}"
    )

persona_rules_df = pd.read_csv(
    persona_rules_path
)

print(
    "\nRules loaded:",
    len(persona_rules_df)
)

print(
    "\nColumns:",
    list(persona_rules_df.columns)
)

print(
    "\nUnique personas:",
    persona_rules_df["Persona"].nunique()
)

display(persona_rules_df)

LOADING PERSONA RULE CATALOGUE

Path:
D:\NudgeIQ\data\processed\persona_rules.csv

Rules loaded: 27

Columns: ['Financial_Level', 'Engagement_Level', 'Profile_Level', 'Persona', 'Tier', 'Priority', 'Action']

Unique personas: 9


,Financial_Level,Engagement_Level,Profile_Level,Persona,Tier,Priority,Action
0,High,High,High,Premium Investor,Tier 1,High,Assign dedicated RM; offer premium/exclusive p...
1,High,High,Medium,Growth Investor,Tier 1,High,Upsell high-growth investment products
2,High,High,Low,Growth Investor,Tier 1,High,Upsell growth products; review profile data qu...
3,High,Medium,High,Growth Investor,Tier 1,High,Upsell high-growth investment products; increa...
4,High,Medium,Medium,Balanced Investor,Tier 2,Medium,Cross-sell balanced portfolio products
5,Medium,High,High,Balanced Investor,Tier 2,Medium,Cross-sell balanced portfolio products
6,Medium,High,Medium,Balanced Investor,Tier 2,Medium,Cross-sell balanced portfolio products
7,Medium,Medium,High,Balanced Investor,Tier 2,Medium,Cross-sell balanced portfolio products
8,Medium,Medium,Medium,General Investor,Tier 2,Medium,Nurture with periodic offers and reviews
9,Medium,Medium,Low,General Investor,Tier 2,Medium,Nurture with periodic offers; monitor profile ...


In [48]:
# ============================================================
# CELL 7 — APPLY PERSONA RULES
# ============================================================

persona_columns = [
    "Financial_Level",
    "Engagement_Level",
    "Profile_Level"
]

# Check that the required columns exist
required_columns = persona_columns + ["Persona"]

missing_columns = [
    col
    for col in required_columns
    if col not in persona_rules_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns in persona_rules.csv: "
        f"{missing_columns}"
    )

# Make sure the rule table has only one rule
# for each behavioural combination
duplicate_rules = persona_rules_df.duplicated(
    subset=persona_columns,
    keep=False
)

if duplicate_rules.any():
    print("WARNING: Duplicate behavioural combinations found:")
    display(
        persona_rules_df[
            duplicate_rules
        ].sort_values(persona_columns)
    )
    raise ValueError(
        "persona_rules.csv contains duplicate rules "
        "for the same behavioural combination."
    )

# ------------------------------------------------------------
# Merge persona rules onto synthetic investors
# ------------------------------------------------------------

synthetic_investors = synthetic_investors.merge(
    persona_rules_df,
    on=persona_columns,
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("PERSONA RULES APPLIED")
print("=" * 70)

print("\nRows:", len(synthetic_investors))

print(
    "Missing Personas:",
    synthetic_investors["Persona"].isna().sum()
)

print(
    "Unique Personas:",
    synthetic_investors["Persona"].nunique()
)

print("\nColumns after merge:")
print(list(synthetic_investors.columns))

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona"
        ]
    ].head(20)
)

PERSONA RULES APPLIED

Rows: 11162
Missing Personas: 0
Unique Personas: 9

Columns after merge:
['Synthetic_Customer_ID', 'Financial_Level', 'Engagement_Level', 'Profile_Level', 'Persona', 'Tier', 'Priority', 'Action']


,Synthetic_Customer_ID,Financial_Level,Engagement_Level,Profile_Level,Persona
0,SYN00001,Medium,Low,Low,Inactive Investor
1,SYN00002,High,Medium,High,Growth Investor
2,SYN00003,Medium,High,Low,Emerging Investor
3,SYN00004,Medium,Medium,Medium,General Investor
4,SYN00005,Low,Medium,Medium,Emerging Investor
5,SYN00006,Low,Medium,Medium,Emerging Investor
6,SYN00007,Low,Low,Medium,Low Engagement User
7,SYN00008,High,Low,Low,Dormant Wealth Holder
8,SYN00009,Medium,Medium,Medium,General Investor
9,SYN00010,Medium,Medium,High,Balanced Investor


In [49]:
# ============================================================
# CELL 8 — PERSONA VALIDATION
# ============================================================

print("=" * 70)
print("PERSONA VALIDATION")
print("=" * 70)

print(
    "\nCustomers:",
    len(synthetic_investors)
)

print(
    "Unique IDs:",
    synthetic_investors[
        "Synthetic_Customer_ID"
    ].nunique()
)

print(
    "Unique Personas:",
    synthetic_investors[
        "Persona"
    ].nunique()
)

print(
    "Missing Personas:",
    synthetic_investors[
        "Persona"
    ].isna().sum()
)

print("\nPersona distribution:")

persona_summary = (
    synthetic_investors[
        "Persona"
    ]
    .value_counts()
    .rename_axis("Persona")
    .reset_index(name="Customers")
)

persona_summary["Percentage"] = (
    persona_summary["Customers"]
    / len(synthetic_investors)
    * 100
).round(2)

display(persona_summary)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(synthetic_investors) == 11162

assert (
    synthetic_investors[
        "Synthetic_Customer_ID"
    ].nunique()
    == 11162
)

assert (
    synthetic_investors[
        "Persona"
    ].isna().sum()
    == 0
)

assert (
    synthetic_investors[
        "Persona"
    ].nunique()
    == 9
)

print("\nPERSONA VALIDATION PASSED")

PERSONA VALIDATION

Customers: 11162
Unique IDs: 11162
Unique Personas: 9
Missing Personas: 0

Persona distribution:


,Persona,Customers,Percentage
0,Inactive Investor,2406,21.56
1,Balanced Investor,2100,18.81
2,General Investor,1902,17.04
3,Low Engagement User,1487,13.32
4,Emerging Investor,1150,10.30
5,Potential Investor,943,8.45
6,Dormant Wealth Holder,539,4.83
7,Growth Investor,441,3.95
8,Premium Investor,194,1.74



PERSONA VALIDATION PASSED


In [50]:
# ============================================================
# STEP 5.1 — SYNTHETIC INVESTOR ATTRIBUTES
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

print("=" * 70)
print("SYNTHETIC INVESTOR ATTRIBUTE GENERATION")
print("=" * 70)


# ------------------------------------------------------------
# ATTRIBUTE RULES
# ------------------------------------------------------------

attribute_rules = {

    "Premium Investor": {
        "Investment_Experience": "High",
        "Risk_Appetite": "Moderate",
        "Investment_Frequency": "Regular",
        "Portfolio_Diversification": "High",
        "Gold_Awareness": "High",
        "Gold_Readiness": "High",
        "Preferred_Gold_Product": "Gold ETF / SGB"
    },

    "Growth Investor": {
        "Investment_Experience": "High",
        "Risk_Appetite": "Aggressive",
        "Investment_Frequency": "Regular",
        "Portfolio_Diversification": "Medium",
        "Gold_Awareness": "Medium",
        "Gold_Readiness": "High",
        "Preferred_Gold_Product": "Gold ETF"
    },

    "Balanced Investor": {
        "Investment_Experience": "Medium",
        "Risk_Appetite": "Moderate",
        "Investment_Frequency": "Regular",
        "Portfolio_Diversification": "High",
        "Gold_Awareness": "Medium",
        "Gold_Readiness": "Medium",
        "Preferred_Gold_Product": "Digital Gold / Gold ETF"
    },

    "General Investor": {
        "Investment_Experience": "Medium",
        "Risk_Appetite": "Moderate",
        "Investment_Frequency": "Occasional",
        "Portfolio_Diversification": "Medium",
        "Gold_Awareness": "Medium",
        "Gold_Readiness": "Medium",
        "Preferred_Gold_Product": "Digital Gold"
    },

    "Potential Investor": {
        "Investment_Experience": "Low",
        "Risk_Appetite": "Moderate",
        "Investment_Frequency": "Occasional",
        "Portfolio_Diversification": "Low",
        "Gold_Awareness": "Low",
        "Gold_Readiness": "Medium",
        "Preferred_Gold_Product": "Digital Gold"
    },

    "Emerging Investor": {
        "Investment_Experience": "Low",
        "Risk_Appetite": "Moderate",
        "Investment_Frequency": "Occasional",
        "Portfolio_Diversification": "Low",
        "Gold_Awareness": "Low",
        "Gold_Readiness": "Low",
        "Preferred_Gold_Product": "Digital Gold"
    },

    "Dormant Wealth Holder": {
        "Investment_Experience": "High",
        "Risk_Appetite": "Conservative",
        "Investment_Frequency": "Rare",
        "Portfolio_Diversification": "Medium",
        "Gold_Awareness": "High",
        "Gold_Readiness": "Medium",
        "Preferred_Gold_Product": "SGB / Gold ETF"
    },

    "Inactive Investor": {
        "Investment_Experience": "Medium",
        "Risk_Appetite": "Conservative",
        "Investment_Frequency": "Rare",
        "Portfolio_Diversification": "Low",
        "Gold_Awareness": "Low",
        "Gold_Readiness": "Low",
        "Preferred_Gold_Product": "Digital Gold"
    },

    "Low Engagement User": {
        "Investment_Experience": "Low",
        "Risk_Appetite": "Conservative",
        "Investment_Frequency": "Rare",
        "Portfolio_Diversification": "Low",
        "Gold_Awareness": "Low",
        "Gold_Readiness": "Low",
        "Preferred_Gold_Product": "Digital Gold"
    }
}


# ------------------------------------------------------------
# APPLY RULES
# ------------------------------------------------------------

for attribute in [
    "Investment_Experience",
    "Risk_Appetite",
    "Investment_Frequency",
    "Portfolio_Diversification",
    "Gold_Awareness",
    "Gold_Readiness",
    "Preferred_Gold_Product"
]:

    synthetic_investors[attribute] = (
        synthetic_investors["Persona"]
        .map(
            lambda x: attribute_rules[x][attribute]
        )
    )


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

attribute_columns = [
    "Investment_Experience",
    "Risk_Appetite",
    "Investment_Frequency",
    "Portfolio_Diversification",
    "Gold_Awareness",
    "Gold_Readiness",
    "Preferred_Gold_Product"
]

print("\nMissing attribute values:")

display(
    synthetic_investors[
        attribute_columns
    ].isna().sum()
)

print("\nSample synthetic investors:")

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Persona",
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level"
        ] + attribute_columns
    ].head(20)
)


# ------------------------------------------------------------
# ASSERTIONS
# ------------------------------------------------------------

assert len(synthetic_investors) == 11162

for col in attribute_columns:
    assert synthetic_investors[col].isna().sum() == 0

print("\n" + "=" * 70)
print("SYNTHETIC INVESTOR ATTRIBUTES CREATED SUCCESSFULLY")
print("=" * 70)

SYNTHETIC INVESTOR ATTRIBUTE GENERATION

Missing attribute values:


Investment_Experience        0
Risk_Appetite                0
Investment_Frequency         0
Portfolio_Diversification    0
Gold_Awareness               0
Gold_Readiness               0
Preferred_Gold_Product       0
dtype: int64


Sample synthetic investors:


,Synthetic_Customer_ID,Persona,Financial_Level,Engagement_Level,Profile_Level,Investment_Experience,Risk_Appetite,Investment_Frequency,Portfolio_Diversification,Gold_Awareness,Gold_Readiness,Preferred_Gold_Product
0,SYN00001,Inactive Investor,Medium,Low,Low,Medium,Conservative,Rare,Low,Low,Low,Digital Gold
1,SYN00002,Growth Investor,High,Medium,High,High,Aggressive,Regular,Medium,Medium,High,Gold ETF
2,SYN00003,Emerging Investor,Medium,High,Low,Low,Moderate,Occasional,Low,Low,Low,Digital Gold
3,SYN00004,General Investor,Medium,Medium,Medium,Medium,Moderate,Occasional,Medium,Medium,Medium,Digital Gold
4,SYN00005,Emerging Investor,Low,Medium,Medium,Low,Moderate,Occasional,Low,Low,Low,Digital Gold
5,SYN00006,Emerging Investor,Low,Medium,Medium,Low,Moderate,Occasional,Low,Low,Low,Digital Gold
6,SYN00007,Low Engagement User,Low,Low,Medium,Low,Conservative,Rare,Low,Low,Low,Digital Gold
7,SYN00008,Dormant Wealth Holder,High,Low,Low,High,Conservative,Rare,Medium,High,Medium,SGB / Gold ETF
8,SYN00009,General Investor,Medium,Medium,Medium,Medium,Moderate,Occasional,Medium,Medium,Medium,Digital Gold
9,SYN00010,Balanced Investor,Medium,Medium,High,Medium,Moderate,Regular,High,Medium,Medium,Digital Gold / Gold ETF



SYNTHETIC INVESTOR ATTRIBUTES CREATED SUCCESSFULLY


In [51]:
# ============================================================
# STEP 5.2 — SYNTHETIC FINANCIAL CAPACITY
# ============================================================

np.random.seed(42)

print("=" * 70)
print("SYNTHETIC FINANCIAL CAPACITY GENERATION")
print("=" * 70)


# ------------------------------------------------------------
# Financial capacity rules
# Based on synthetic behavioural levels + persona
# ------------------------------------------------------------

financial_capacity_rules = {

    "Low": {
        "min": 1000,
        "max": 10000
    },

    "Medium": {
        "min": 10000,
        "max": 100000
    },

    "High": {
        "min": 100000,
        "max": 500000
    }
}


# ------------------------------------------------------------
# Generate synthetic investable capacity
# ------------------------------------------------------------

def generate_capacity(level):

    rule = financial_capacity_rules[level]

    return np.random.uniform(
        rule["min"],
        rule["max"]
    )


synthetic_investors[
    "Estimated_Investment_Capacity"
] = (
    synthetic_investors[
        "Financial_Level"
    ]
    .map(generate_capacity)
    .round(2)
)


# ------------------------------------------------------------
# Generate synthetic monthly investment capacity
# ------------------------------------------------------------

capacity_ratio = {
    "Low": 0.05,
    "Medium": 0.08,
    "High": 0.12
}

synthetic_investors[
    "Estimated_Monthly_Investment_Capacity"
] = (
    synthetic_investors[
        "Estimated_Investment_Capacity"
    ]
    *
    synthetic_investors[
        "Financial_Level"
    ].map(capacity_ratio)
).round(2)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nFinancial Level distribution:")

display(
    synthetic_investors[
        "Financial_Level"
    ].value_counts()
    .rename_axis("Financial_Level")
    .reset_index(name="Customers")
)


print("\nSynthetic financial capacity:")

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Financial_Level",
            "Estimated_Investment_Capacity",
            "Estimated_Monthly_Investment_Capacity"
        ]
    ].head(20)
)


print("\nSummary:")

display(
    synthetic_investors[
        [
            "Estimated_Investment_Capacity",
            "Estimated_Monthly_Investment_Capacity"
        ]
    ].describe()
)


assert (
    synthetic_investors[
        "Estimated_Investment_Capacity"
    ].isna().sum()
    == 0
)

assert (
    synthetic_investors[
        "Estimated_Monthly_Investment_Capacity"
    ].isna().sum()
    == 0
)

print("\n" + "=" * 70)
print("FINANCIAL CAPACITY CREATED SUCCESSFULLY")
print("=" * 70)

SYNTHETIC FINANCIAL CAPACITY GENERATION

Financial Level distribution:


,Financial_Level,Customers
0,Medium,6333
1,Low,3127
2,High,1702



Synthetic financial capacity:


,Synthetic_Customer_ID,Financial_Level,Estimated_Investment_Capacity,Estimated_Monthly_Investment_Capacity
0,SYN00001,Medium,43708.61,3496.69
1,SYN00002,High,480285.72,57634.29
2,SYN00003,Medium,75879.45,6070.36
3,SYN00004,Medium,63879.26,5110.34
4,SYN00005,Low,2404.17,120.21
5,SYN00006,Low,2403.95,120.20
6,SYN00007,Low,1522.75,76.14
7,SYN00008,High,446470.46,53576.46
8,SYN00009,Medium,64100.35,5128.03
9,SYN00010,Medium,73726.53,5898.12



Summary:


,Estimated_Investment_Capacity,Estimated_Monthly_Investment_Capacity
count,11162.000000,11162.000000
mean,106220.162085,11335.729954
std,156220.323703,19182.058040
min,1000.100000,50.000000
25%,3226.137500,161.310000
50%,54522.610000,4361.810000
75%,76801.615000,6144.127500
max,499887.070000,59986.450000



FINANCIAL CAPACITY CREATED SUCCESSFULLY


In [52]:
# ============================================================
# STEP 5.3 — SYNTHETIC INVESTMENT BEHAVIOUR
# ============================================================

np.random.seed(42)

print("=" * 70)
print("SYNTHETIC INVESTMENT BEHAVIOUR GENERATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Investment frequency
# ------------------------------------------------------------

frequency_map = {
    "Premium Investor": "Regular",
    "Growth Investor": "Regular",
    "Balanced Investor": "Regular",
    "General Investor": "Occasional",
    "Potential Investor": "Occasional",
    "Emerging Investor": "Occasional",
    "Dormant Wealth Holder": "Rare",
    "Inactive Investor": "Rare",
    "Low Engagement User": "Rare"
}

synthetic_investors["Investment_Frequency"] = (
    synthetic_investors["Persona"]
    .map(frequency_map)
)


# ------------------------------------------------------------
# 2. Portfolio diversification
# ------------------------------------------------------------

diversification_map = {
    "Premium Investor": "High",
    "Growth Investor": "Medium",
    "Balanced Investor": "High",
    "General Investor": "Medium",
    "Potential Investor": "Low",
    "Emerging Investor": "Low",
    "Dormant Wealth Holder": "Medium",
    "Inactive Investor": "Low",
    "Low Engagement User": "Low"
}

synthetic_investors["Portfolio_Diversification"] = (
    synthetic_investors["Persona"]
    .map(diversification_map)
)


# ------------------------------------------------------------
# 3. Gold allocation potential
# ------------------------------------------------------------

gold_allocation_map = {
    "Premium Investor": "High",
    "Growth Investor": "High",
    "Balanced Investor": "Medium",
    "General Investor": "Medium",
    "Potential Investor": "Low",
    "Emerging Investor": "Low",
    "Dormant Wealth Holder": "Medium",
    "Inactive Investor": "Low",
    "Low Engagement User": "Low"
}

synthetic_investors["Gold_Allocation_Potential"] = (
    synthetic_investors["Persona"]
    .map(gold_allocation_map)
)


# ------------------------------------------------------------
# 4. Gold awareness score
# ------------------------------------------------------------

awareness_score = {
    "Low": 0.25,
    "Medium": 0.60,
    "High": 0.90
}

synthetic_investors["Gold_Awareness_Score"] = (
    synthetic_investors["Gold_Awareness"]
    .map(awareness_score)
)


# ------------------------------------------------------------
# 5. Gold readiness score
# ------------------------------------------------------------

readiness_score = {
    "Low": 0.25,
    "Medium": 0.60,
    "High": 0.90
}

synthetic_investors["Gold_Readiness_Score"] = (
    synthetic_investors["Gold_Readiness"]
    .map(readiness_score)
)


# ------------------------------------------------------------
# 6. Investment experience score
# ------------------------------------------------------------

experience_score = {
    "Low": 0.25,
    "Medium": 0.60,
    "High": 0.90
}

synthetic_investors["Investment_Experience_Score"] = (
    synthetic_investors["Investment_Experience"]
    .map(experience_score)
)


# ------------------------------------------------------------
# 7. Behavioural engagement score
# ------------------------------------------------------------

engagement_score = {
    "Low": 0.25,
    "Medium": 0.60,
    "High": 0.90
}

synthetic_investors["Behavioural_Engagement_Score"] = (
    synthetic_investors["Engagement_Level"]
    .map(engagement_score)
)


# ------------------------------------------------------------
# 8. Financial capacity score
# ------------------------------------------------------------

financial_score = {
    "Low": 0.25,
    "Medium": 0.60,
    "High": 0.90
}

synthetic_investors["Financial_Capacity_Score"] = (
    synthetic_investors["Financial_Level"]
    .map(financial_score)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

behaviour_columns = [
    "Investment_Frequency",
    "Portfolio_Diversification",
    "Gold_Allocation_Potential",
    "Gold_Awareness_Score",
    "Gold_Readiness_Score",
    "Investment_Experience_Score",
    "Behavioural_Engagement_Score",
    "Financial_Capacity_Score"
]

print("\nMissing values:")

display(
    synthetic_investors[
        behaviour_columns
    ].isna().sum()
)


print("\nSample:")

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Persona",
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Investment_Frequency",
            "Portfolio_Diversification",
            "Gold_Allocation_Potential",
            "Gold_Awareness_Score",
            "Gold_Readiness_Score"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

for col in behaviour_columns:
    assert (
        synthetic_investors[col].isna().sum()
        == 0
    )

print("\n" + "=" * 70)
print("SYNTHETIC INVESTMENT BEHAVIOUR CREATED")
print("=" * 70)

SYNTHETIC INVESTMENT BEHAVIOUR GENERATION

Missing values:


Investment_Frequency            0
Portfolio_Diversification       0
Gold_Allocation_Potential       0
Gold_Awareness_Score            0
Gold_Readiness_Score            0
Investment_Experience_Score     0
Behavioural_Engagement_Score    0
Financial_Capacity_Score        0
dtype: int64


Sample:


,Synthetic_Customer_ID,Persona,Financial_Level,Engagement_Level,Profile_Level,Investment_Frequency,Portfolio_Diversification,Gold_Allocation_Potential,Gold_Awareness_Score,Gold_Readiness_Score
0,SYN00001,Inactive Investor,Medium,Low,Low,Rare,Low,Low,0.25,0.25
1,SYN00002,Growth Investor,High,Medium,High,Regular,Medium,High,0.60,0.90
2,SYN00003,Emerging Investor,Medium,High,Low,Occasional,Low,Low,0.25,0.25
3,SYN00004,General Investor,Medium,Medium,Medium,Occasional,Medium,Medium,0.60,0.60
4,SYN00005,Emerging Investor,Low,Medium,Medium,Occasional,Low,Low,0.25,0.25
5,SYN00006,Emerging Investor,Low,Medium,Medium,Occasional,Low,Low,0.25,0.25
6,SYN00007,Low Engagement User,Low,Low,Medium,Rare,Low,Low,0.25,0.25
7,SYN00008,Dormant Wealth Holder,High,Low,Low,Rare,Medium,Medium,0.90,0.60
8,SYN00009,General Investor,Medium,Medium,Medium,Occasional,Medium,Medium,0.60,0.60
9,SYN00010,Balanced Investor,Medium,Medium,High,Regular,High,Medium,0.60,0.60



SYNTHETIC INVESTMENT BEHAVIOUR CREATED


In [53]:
# ============================================================
# STEP 5.4 — GOLD OPPORTUNITY SCORE
# ============================================================

print("=" * 70)
print("GOLD OPPORTUNITY SCORE")
print("=" * 70)


# ------------------------------------------------------------
# Component weights
# ------------------------------------------------------------

GOLD_WEIGHTS = {
    "Gold_Readiness_Score": 0.30,
    "Financial_Capacity_Score": 0.25,
    "Behavioural_Engagement_Score": 0.20,
    "Gold_Awareness_Score": 0.15,
    "Investment_Experience_Score": 0.10
}


# ------------------------------------------------------------
# Calculate weighted components
# ------------------------------------------------------------

synthetic_investors["Gold_Readiness_Component"] = (
    synthetic_investors["Gold_Readiness_Score"]
    * GOLD_WEIGHTS["Gold_Readiness_Score"]
)

synthetic_investors["Financial_Capacity_Component"] = (
    synthetic_investors["Financial_Capacity_Score"]
    * GOLD_WEIGHTS["Financial_Capacity_Score"]
)

synthetic_investors["Engagement_Component"] = (
    synthetic_investors["Behavioural_Engagement_Score"]
    * GOLD_WEIGHTS["Behavioural_Engagement_Score"]
)

synthetic_investors["Gold_Awareness_Component"] = (
    synthetic_investors["Gold_Awareness_Score"]
    * GOLD_WEIGHTS["Gold_Awareness_Score"]
)

synthetic_investors["Investment_Experience_Component"] = (
    synthetic_investors["Investment_Experience_Score"]
    * GOLD_WEIGHTS["Investment_Experience_Score"]
)


# ------------------------------------------------------------
# Final Gold Opportunity Score
# ------------------------------------------------------------

synthetic_investors["Gold_Opportunity_Score"] = (
    synthetic_investors[
        "Gold_Readiness_Component"
    ]
    + synthetic_investors[
        "Financial_Capacity_Component"
    ]
    + synthetic_investors[
        "Engagement_Component"
    ]
    + synthetic_investors[
        "Gold_Awareness_Component"
    ]
    + synthetic_investors[
        "Investment_Experience_Component"
    ]
).round(4)


# ------------------------------------------------------------
# Score validation
# ------------------------------------------------------------

print("\nScore statistics:")

display(
    synthetic_investors[
        "Gold_Opportunity_Score"
    ].describe()
)


print("\nMinimum score:",
      synthetic_investors[
          "Gold_Opportunity_Score"
      ].min())

print("Maximum score:",
      synthetic_investors[
          "Gold_Opportunity_Score"
      ].max())

print("Average score:",
      synthetic_investors[
          "Gold_Opportunity_Score"
      ].mean())


# ------------------------------------------------------------
# Sample
# ------------------------------------------------------------

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Persona",
            "Gold_Readiness_Score",
            "Financial_Capacity_Score",
            "Behavioural_Engagement_Score",
            "Gold_Awareness_Score",
            "Investment_Experience_Score",
            "Gold_Opportunity_Score"
        ]
    ]
    .sort_values(
        "Gold_Opportunity_Score",
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    synthetic_investors[
        "Gold_Opportunity_Score"
    ].isna().sum()
    == 0
)

assert (
    synthetic_investors[
        "Gold_Opportunity_Score"
    ].between(0, 1).all()
)

print("\n" + "=" * 70)
print("GOLD OPPORTUNITY SCORE CREATED SUCCESSFULLY")
print("=" * 70)

GOLD OPPORTUNITY SCORE

Score statistics:


count    11162.000000
mean         0.498342
std          0.164955
min          0.250000
25%          0.372500
50%          0.485000
75%          0.600000
max          0.900000
Name: Gold_Opportunity_Score, dtype: float64


Minimum score: 0.25
Maximum score: 0.9
Average score: 0.49834191901092995


,Synthetic_Customer_ID,Persona,Gold_Readiness_Score,Financial_Capacity_Score,Behavioural_Engagement_Score,Gold_Awareness_Score,Investment_Experience_Score,Gold_Opportunity_Score
11139,SYN11140,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
11099,SYN11100,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
10949,SYN10950,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
10703,SYN10704,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
10648,SYN10649,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
932,SYN00933,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
847,SYN00848,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
810,SYN00811,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
705,SYN00706,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9
1061,SYN01062,Premium Investor,0.9,0.9,0.9,0.9,0.9,0.9



GOLD OPPORTUNITY SCORE CREATED SUCCESSFULLY


In [54]:
# ============================================================
# STEP 5.5 — GOLD OPPORTUNITY BANDS
# ============================================================

print("=" * 70)
print("GOLD OPPORTUNITY BANDS")
print("=" * 70)


# ------------------------------------------------------------
# Determine data-driven thresholds
# ------------------------------------------------------------

low_threshold = (
    synthetic_investors[
        "Gold_Opportunity_Score"
    ].quantile(0.25)
)

high_threshold = (
    synthetic_investors[
        "Gold_Opportunity_Score"
    ].quantile(0.75)
)


print(
    f"\nLow / Medium threshold: "
    f"{low_threshold:.3f}"
)

print(
    f"Medium / High threshold: "
    f"{high_threshold:.3f}"
)


# ------------------------------------------------------------
# Assign opportunity band
# ------------------------------------------------------------

def assign_opportunity_band(score):

    if score >= high_threshold:
        return "HIGH"

    elif score >= low_threshold:
        return "MEDIUM"

    else:
        return "LOW"


synthetic_investors[
    "Gold_Opportunity_Band"
] = (
    synthetic_investors[
        "Gold_Opportunity_Score"
    ]
    .apply(assign_opportunity_band)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

opportunity_summary = (
    synthetic_investors[
        "Gold_Opportunity_Band"
    ]
    .value_counts()
    .rename_axis(
        "Gold_Opportunity_Band"
    )
    .reset_index(
        name="Customers"
    )
)

opportunity_summary["Percentage"] = (
    opportunity_summary["Customers"]
    / len(synthetic_investors)
    * 100
).round(2)


print("\nOpportunity distribution:")

display(
    opportunity_summary
)


# ------------------------------------------------------------
# Score statistics by band
# ------------------------------------------------------------

band_statistics = (
    synthetic_investors
    .groupby(
        "Gold_Opportunity_Band"
    )
    ["Gold_Opportunity_Score"]
    .agg(
        Customers="count",
        Minimum="min",
        Average="mean",
        Maximum="max"
    )
    .reset_index()
)

display(
    band_statistics
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    synthetic_investors[
        "Gold_Opportunity_Band"
    ].isna().sum()
    == 0
)

assert set(
    synthetic_investors[
        "Gold_Opportunity_Band"
    ].unique()
) == {
    "LOW",
    "MEDIUM",
    "HIGH"
}

print(
    "\n" + "=" * 70
)

print(
    "GOLD OPPORTUNITY BANDS CREATED SUCCESSFULLY"
)

print("=" * 70)

GOLD OPPORTUNITY BANDS

Low / Medium threshold: 0.372
Medium / High threshold: 0.600

Opportunity distribution:


,Gold_Opportunity_Band,Customers,Percentage
0,HIGH,5176,46.37
1,MEDIUM,3934,35.24
2,LOW,2052,18.38


,Gold_Opportunity_Band,Customers,Minimum,Average,Maximum
0,HIGH,5176,0.6000,0.656854,0.900
1,LOW,2052,0.2500,0.283021,0.320
2,MEDIUM,3934,0.3725,0.402099,0.485



GOLD OPPORTUNITY BANDS CREATED SUCCESSFULLY


In [55]:
# ============================================================
# STEP 5.6 — NEXT BEST ACTION ENGINE
# ============================================================

print("=" * 70)
print("NEXT BEST ACTION ENGINE")
print("=" * 70)


# ------------------------------------------------------------
# Decision function
# ------------------------------------------------------------

def determine_next_best_action(row):

    persona = row["Persona"]
    band = row["Gold_Opportunity_Band"]
    engagement = row["Engagement_Level"]
    readiness = row["Gold_Readiness"]

    # --------------------------------------------------------
    # LOW OPPORTUNITY
    # --------------------------------------------------------

    if band == "LOW":

        return pd.Series({
            "Next_Best_Action":
                "Low-friction gold awareness content",

            "Recommended_Channel":
                "Low-cost Digital",

            "Nudge_Intensity":
                "Low",

            "Business_Objective":
                "Build gold awareness",

            "Decision_Rationale":
                "Low current gold opportunity; prioritize education over conversion"
        })


    # --------------------------------------------------------
    # MEDIUM OPPORTUNITY
    # --------------------------------------------------------

    if band == "MEDIUM":

        if readiness == "Low":

            return pd.Series({
                "Next_Best_Action":
                    "Gold education and portfolio-awareness journey",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "Medium",

                "Business_Objective":
                    "Build gold consideration",

                "Decision_Rationale":
                    "Moderate opportunity but low readiness requires education"
            })

        elif engagement == "Low":

            return pd.Series({
                "Next_Best_Action":
                    "Re-engagement followed by gold education",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "Medium",

                "Business_Objective":
                    "Re-engage before conversion",

                "Decision_Rationale":
                    "Moderate opportunity but low engagement creates friction"
            })

        else:

            return pd.Series({
                "Next_Best_Action":
                    "Personalized gold diversification recommendation",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "Medium",

                "Business_Objective":
                    "Increase gold consideration",

                "Decision_Rationale":
                    "Moderate opportunity with sufficient engagement/readiness"
            })


    # --------------------------------------------------------
    # HIGH OPPORTUNITY
    # --------------------------------------------------------

    if band == "HIGH":

        # Premium / high-value customers
        if persona in [
            "Premium Investor",
            "Growth Investor",
            "Dormant Wealth Holder"
        ]:

            return pd.Series({
                "Next_Best_Action":
                    "Personalized strategic gold recommendation",

                "Recommended_Channel":
                    "Relationship Manager / RM + Digital",

                "Nudge_Intensity":
                    "High",

                "Business_Objective":
                    "Drive personalized gold investment consideration",

                "Decision_Rationale":
                    "High opportunity combined with high-value investor profile"
            })


        # Highly engaged investors
        elif engagement == "High":

            return pd.Series({
                "Next_Best_Action":
                    "Gold diversification recommendation",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "High",

                "Business_Objective":
                    "Convert gold interest into consideration",

                "Decision_Rationale":
                    "High opportunity and strong behavioural engagement"
            })


        # Medium engagement / readiness
        elif readiness == "High":

            return pd.Series({
                "Next_Best_Action":
                    "Gold conversion-focused digital nudge",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "Medium-High",

                "Business_Objective":
                    "Convert gold readiness into action",

                "Decision_Rationale":
                    "High opportunity and strong investment readiness"
            })


        else:

            return pd.Series({
                "Next_Best_Action":
                    "Personalized gold consideration message",

                "Recommended_Channel":
                    "Digital",

                "Nudge_Intensity":
                    "Medium-High",

                "Business_Objective":
                    "Increase gold consideration",

                "Decision_Rationale":
                    "High opportunity but conversion barriers remain"
            })


# ------------------------------------------------------------
# Apply decision engine
# ------------------------------------------------------------

next_best_action = (
    synthetic_investors.apply(
        determine_next_best_action,
        axis=1
    )
)


# ------------------------------------------------------------
# Add outputs
# ------------------------------------------------------------

synthetic_investors = pd.concat(
    [
        synthetic_investors,
        next_best_action
    ],
    axis=1
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nMissing values:")

display(
    synthetic_investors[
        [
            "Next_Best_Action",
            "Recommended_Channel",
            "Nudge_Intensity",
            "Business_Objective",
            "Decision_Rationale"
        ]
    ].isna().sum()
)


print("\nNext Best Action distribution:")

nba_summary = (
    synthetic_investors[
        "Next_Best_Action"
    ]
    .value_counts()
    .rename_axis("Next_Best_Action")
    .reset_index(name="Customers")
)

nba_summary["Percentage"] = (
    nba_summary["Customers"]
    / len(synthetic_investors)
    * 100
).round(2)

display(nba_summary)


print("\nChannel distribution:")

display(
    synthetic_investors[
        "Recommended_Channel"
    ]
    .value_counts()
)


print("\nSample decisions:")

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Persona",
            "Gold_Opportunity_Score",
            "Gold_Opportunity_Band",
            "Gold_Readiness",
            "Engagement_Level",
            "Next_Best_Action",
            "Recommended_Channel",
            "Nudge_Intensity"
        ]
    ]
    .sort_values(
        "Gold_Opportunity_Score",
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

nba_columns = [
    "Next_Best_Action",
    "Recommended_Channel",
    "Nudge_Intensity",
    "Business_Objective",
    "Decision_Rationale"
]

for col in nba_columns:

    assert (
        synthetic_investors[col].isna().sum()
        == 0
    )


print("\n" + "=" * 70)
print("NEXT BEST ACTION ENGINE CREATED SUCCESSFULLY")
print("=" * 70)

NEXT BEST ACTION ENGINE

Missing values:


Next_Best_Action       0
Recommended_Channel    0
Nudge_Intensity        0
Business_Objective     0
Decision_Rationale     0
dtype: int64


Next Best Action distribution:


,Next_Best_Action,Customers,Percentage
0,Gold education and portfolio-awareness journey,2991,26.80
1,Personalized gold consideration message,2864,25.66
2,Low-friction gold awareness content,2052,18.38
3,Personalized strategic gold recommendation,1174,10.52
4,Gold diversification recommendation,1138,10.20
5,Personalized gold diversification recommendation,630,5.64
6,Re-engagement followed by gold education,313,2.80



Channel distribution:


Recommended_Channel
Digital                                7936
Low-cost Digital                       2052
Relationship Manager / RM + Digital    1174
Name: count, dtype: int64


Sample decisions:


,Synthetic_Customer_ID,Persona,Gold_Opportunity_Score,Gold_Opportunity_Band,Gold_Readiness,Engagement_Level,Next_Best_Action,Recommended_Channel,Nudge_Intensity
11139,SYN11140,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
11099,SYN11100,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
10949,SYN10950,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
10703,SYN10704,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
10648,SYN10649,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
932,SYN00933,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
847,SYN00848,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
810,SYN00811,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
705,SYN00706,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High
1061,SYN01062,Premium Investor,0.9,HIGH,High,High,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High



NEXT BEST ACTION ENGINE CREATED SUCCESSFULLY


In [56]:
# ============================================================
# STEP 5.7 — GOLD NUDGE MESSAGE ENGINE
# ============================================================

print("=" * 70)
print("GOLD NUDGE MESSAGE ENGINE")
print("=" * 70)


# ------------------------------------------------------------
# Message catalogue
# ------------------------------------------------------------

message_map = {

    "Low-friction gold awareness content":
        "Curious about gold investing? Learn how gold can complement a diversified investment portfolio.",

    "Gold education and portfolio-awareness journey":
        "Explore how gold works as an investment and how it may complement your existing financial goals.",

    "Re-engagement followed by gold education":
        "It's a good time to review your investment options. Explore how gold could complement your portfolio.",

    "Personalized gold consideration message":
        "Explore gold as an additional investment option and see how it could fit into your overall portfolio.",

    "Gold diversification recommendation":
        "Looking to diversify your investments? Explore gold as another asset that could complement your portfolio.",

    "Personalized gold diversification recommendation":
        "Consider adding gold to your portfolio as another diversification option aligned with your investment profile.",

    "Personalized strategic gold recommendation":
        "Explore whether a strategic allocation to gold could complement your broader investment portfolio."
}


# ------------------------------------------------------------
# Assign message
# ------------------------------------------------------------

synthetic_investors["Gold_Nudge_Message"] = (
    synthetic_investors[
        "Next_Best_Action"
    ].map(message_map)
)


# ------------------------------------------------------------
# Message validation
# ------------------------------------------------------------

missing_messages = (
    synthetic_investors[
        "Gold_Nudge_Message"
    ].isna().sum()
)

print("\nMissing messages:", missing_messages)


# ------------------------------------------------------------
# Display sample
# ------------------------------------------------------------

display(
    synthetic_investors[
        [
            "Synthetic_Customer_ID",
            "Persona",
            "Gold_Opportunity_Band",
            "Gold_Opportunity_Score",
            "Next_Best_Action",
            "Recommended_Channel",
            "Nudge_Intensity",
            "Gold_Nudge_Message"
        ]
    ]
    .sort_values(
        "Gold_Opportunity_Score",
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# Message coverage
# ------------------------------------------------------------

message_summary = (
    synthetic_investors[
        [
            "Next_Best_Action",
            "Gold_Nudge_Message"
        ]
    ]
    .drop_duplicates()
)

print("\nAction → Message coverage:")

display(
    message_summary
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert missing_messages == 0

assert (
    synthetic_investors[
        "Gold_Nudge_Message"
    ].astype(str).str.len().gt(0).all()
)


print("\n" + "=" * 70)
print("GOLD NUDGE MESSAGE ENGINE CREATED SUCCESSFULLY")
print("=" * 70)

GOLD NUDGE MESSAGE ENGINE

Missing messages: 0


,Synthetic_Customer_ID,Persona,Gold_Opportunity_Band,Gold_Opportunity_Score,Next_Best_Action,Recommended_Channel,Nudge_Intensity,Gold_Nudge_Message
11139,SYN11140,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
11099,SYN11100,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
10949,SYN10950,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
10703,SYN10704,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
10648,SYN10649,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
932,SYN00933,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
847,SYN00848,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
810,SYN00811,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
705,SYN00706,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...
1061,SYN01062,Premium Investor,HIGH,0.9,Personalized strategic gold recommendation,Relationship Manager / RM + Digital,High,Explore whether a strategic allocation to gold...



Action → Message coverage:


,Next_Best_Action,Gold_Nudge_Message
0,Gold education and portfolio-awareness journey,Explore how gold works as an investment and ho...
1,Personalized strategic gold recommendation,Explore whether a strategic allocation to gold...
3,Personalized gold consideration message,Explore gold as an additional investment optio...
4,Low-friction gold awareness content,Curious about gold investing? Learn how gold c...
12,Gold diversification recommendation,Looking to diversify your investments? Explore...
14,Personalized gold diversification recommendation,Consider adding gold to your portfolio as anot...
89,Re-engagement followed by gold education,It's a good time to review your investment opt...



GOLD NUDGE MESSAGE ENGINE CREATED SUCCESSFULLY


In [57]:
# ============================================================
# STEP 5.8 — RECOMMENDED MESSAGING COHORTS
# ============================================================

print("=" * 70)
print("RECOMMENDED MESSAGING COHORTS")
print("=" * 70)


def assign_messaging_cohort(row):

    band = row["Gold_Opportunity_Band"]
    readiness = row["Gold_Readiness"]
    engagement = row["Engagement_Level"]

    # LOW OPPORTUNITY
    if band == "LOW":
        return "AWARENESS_COHORT"

    # MEDIUM OPPORTUNITY
    elif band == "MEDIUM":

        if readiness == "Low":
            return "EDUCATION_COHORT"

        elif engagement == "Low":
            return "REENGAGEMENT_COHORT"

        else:
            return "CONSIDERATION_COHORT"

    # HIGH OPPORTUNITY
    elif band == "HIGH":

        if row["Persona"] in [
            "Premium Investor",
            "Growth Investor",
            "Dormant Wealth Holder"
        ]:
            return "STRATEGIC_COHORT"

        elif readiness == "High":
            return "CONVERSION_COHORT"

        else:
            return "PRIORITY_CONSIDERATION_COHORT"


synthetic_investors[
    "Recommended_Messaging_Cohort"
] = (
    synthetic_investors.apply(
        assign_messaging_cohort,
        axis=1
    )
)


# ------------------------------------------------------------
# Cohort distribution
# ------------------------------------------------------------

cohort_summary = (
    synthetic_investors[
        "Recommended_Messaging_Cohort"
    ]
    .value_counts()
    .rename_axis(
        "Recommended_Messaging_Cohort"
    )
    .reset_index(
        name="Customers"
    )
)

cohort_summary["Percentage"] = (
    cohort_summary["Customers"]
    / len(synthetic_investors)
    * 100
).round(2)


print("\nCohort distribution:")

display(
    cohort_summary
)


# ------------------------------------------------------------
# Cohort × Opportunity Band
# ------------------------------------------------------------

cohort_band = pd.crosstab(
    synthetic_investors[
        "Recommended_Messaging_Cohort"
    ],
    synthetic_investors[
        "Gold_Opportunity_Band"
    ]
)

print("\nCohort × Opportunity Band:")

display(cohort_band)


# ------------------------------------------------------------
# Cohort × Channel
# ------------------------------------------------------------

cohort_channel = pd.crosstab(
    synthetic_investors[
        "Recommended_Messaging_Cohort"
    ],
    synthetic_investors[
        "Recommended_Channel"
    ]
)

print("\nCohort × Channel:")

display(cohort_channel)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    synthetic_investors[
        "Recommended_Messaging_Cohort"
    ].isna().sum()
    == 0
)

print("\n" + "=" * 70)
print("RECOMMENDED MESSAGING COHORTS CREATED")
print("=" * 70)

RECOMMENDED MESSAGING COHORTS

Cohort distribution:


,Recommended_Messaging_Cohort,Customers,Percentage
0,PRIORITY_CONSIDERATION_COHORT,4002,35.85
1,EDUCATION_COHORT,2991,26.80
2,AWARENESS_COHORT,2052,18.38
3,STRATEGIC_COHORT,1174,10.52
4,CONSIDERATION_COHORT,630,5.64
5,REENGAGEMENT_COHORT,313,2.80



Cohort × Opportunity Band:


Gold_Opportunity_Band,HIGH,LOW,MEDIUM
Recommended_Messaging_Cohort,,,
AWARENESS_COHORT,0,2052,0
CONSIDERATION_COHORT,0,0,630
EDUCATION_COHORT,0,0,2991
PRIORITY_CONSIDERATION_COHORT,4002,0,0
REENGAGEMENT_COHORT,0,0,313
STRATEGIC_COHORT,1174,0,0



Cohort × Channel:


Recommended_Channel,Digital,Low-cost Digital,Relationship Manager / RM + Digital
Recommended_Messaging_Cohort,,,
AWARENESS_COHORT,0,2052,0
CONSIDERATION_COHORT,630,0,0
EDUCATION_COHORT,2991,0,0
PRIORITY_CONSIDERATION_COHORT,4002,0,0
REENGAGEMENT_COHORT,313,0,0
STRATEGIC_COHORT,0,0,1174



RECOMMENDED MESSAGING COHORTS CREATED
